*AI-generated draft (Claude, Anthropic) — for review. All parameters derive from version-controlled scripts/data.*

<span style="font-family: 'Courier New', monospace;">

# 39 · Resolve the 8 unsure 2017 frames

In the salvage pass you marked **8 of the 31 2017 recordings `unsure`**. This isolates just those 8 for one decisive look, so the final 2017 usable-Scene-1 count is clean. These are the Aug–Oct slots plus `20170320`.

**The rubric (unchanged, `scene_sorting/scene1_rubric.md`):** Scene 1 = the **Mushroom sulfide TOWER, face-on, zoomed in, centered/upright, vent orifice visible**. NOT wide/survey, edge-on/behind, or generic vent-community texture.

**Goal: a decision.** Zoom in, take your time, and commit to **✓ Scene 1** (set the time) or **✗ Not Scene 1**. Only fall back to **? Unsure** if it is genuinely undecidable after a careful look — the point of this pass is to clear the unsures, not re-log them.

Isolated dir `validation/salvage_2017_unsure/` — blind to Lyr's calls, touches nothing else. Resumes where you left off. **Kernel: scaleworm.** Restart & Run All, then click. When done, tell Claude — it folds these into the 2017 tally and updates the accepted-frame list.

</span>

In [ ]:
import csv
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

%matplotlib inline

BASE = Path("/home/jovyan/scaleworm-student-lab/validation/salvage_2017_unsure")
MANIFEST = BASE / "resolve_manifest.csv"
DONE = {"scene1", "not_scene1", "unsure"}


def read_manifest():
    with MANIFEST.open(newline="") as fh:
        r = csv.DictReader(fh)
        return list(r), r.fieldnames


def write_manifest(rows, fields):
    with MANIFEST.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


ROWS, FIELDS = read_manifest()
n_done = sum(1 for r in ROWS if r["mj_decision"] in DONE)
print(f"{len(ROWS)} formerly-unsure 2017 recordings  ({n_done} done, {len(ROWS) - n_done} to resolve).")
print("Apply YOUR Scene-1 rubric. Aim for a decision. Click a button, then Save & Next.")

In [ ]:
class Resolver:
    """Decisive re-look at the 8 formerly-unsure 2017 frames; writes mj_decision back."""

    def __init__(self, rows, fields):
        self.rows, self.fields = rows, fields
        self.pos = next((i for i, r in enumerate(rows) if r["mj_decision"] not in DONE), 0)
        self.pending = None

        def mk(desc, style=""):
            return widgets.Button(description=desc, button_style=style,
                                  layout=widgets.Layout(width="auto"))

        self.b_s1 = mk("✓ Scene 1", "success")
        self.b_no = mk("✗ Not Scene 1", "danger")
        self.b_uns = mk("? Unsure", "warning")
        self.time = widgets.IntText(value=0, description="t (s):",
                                    layout=widgets.Layout(width="140px"))
        self.b_save = mk("Save & Next →", "primary")
        self.b_prev = mk("← Back")
        self.status = widgets.HTML()
        self.out = widgets.Output()

        self.b_s1.on_click(lambda _: self.stage("scene1"))
        self.b_no.on_click(lambda _: self.stage("not_scene1"))
        self.b_uns.on_click(lambda _: self.stage("unsure"))
        self.b_save.on_click(self.save_next)
        self.b_prev.on_click(self.back)

        controls = widgets.HBox([self.b_s1, self.b_no, self.b_uns, self.time])
        nav = widgets.HBox([self.b_prev, self.b_save])
        display(widgets.VBox([self.status, controls, nav, self.out]))
        self.show()

    def stage(self, decision):
        self.pending = decision
        r = self.rows[self.pos]
        t = f" @ t={self.time.value}s" if decision == "scene1" else ""
        self.status.value = (f"<b>{r['frame_id']}</b> &mdash; staged: <b>{decision}</b>{t}. "
                             f"Adjust t if Scene 1, then Save & Next.")

    def show(self):
        self.pending = None
        with self.out:
            self.out.clear_output(wait=True)
            if self.pos >= len(self.rows):
                print("All done — tell Claude to fold these into the 2017 tally.")
                return
            r = self.rows[self.pos]
            done = sum(1 for x in self.rows if x["mj_decision"] in DONE)
            self.status.value = (f"<b>{r['frame_id']}</b> &mdash; "
                                 f"{self.pos + 1}/{len(self.rows)}  ({done} done)")
            img = mpimg.imread(BASE / r["sheet_path"])
            fig, ax = plt.subplots(figsize=(15, 8))
            ax.imshow(img); ax.axis("off")
            ax.set_title(r["frame_id"], fontsize=10)
            plt.show()

    def save_next(self, _):
        if self.pos >= len(self.rows):
            return
        if self.pending is None:
            self.status.value = "<b style='color:red'>Pick ✓ / ✗ / ? first.</b>"
            return
        r = self.rows[self.pos]
        r["mj_decision"] = self.pending
        r["mj_scene1_time_s"] = str(self.time.value) if self.pending == "scene1" else ""
        r["decided_at_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
        r["notes"] = "resolved (was unsure)"
        write_manifest(self.rows, self.fields)
        self.pos += 1
        self.time.value = 0
        self.show()

    def back(self, _):
        if self.pos > 0:
            self.pos -= 1
            self.time.value = 0
            self.show()


R = Resolver(ROWS, FIELDS)